## Regression Returns

Description:     
In this approach, we treat the next-bar (or multi-bar) return as a continuous variable and use a regression model (e.g., RandomForestRegressor) to predict it. Positive predicted returns imply a potential buy signal, negative imply a sell, and near-zero might mean no trade. This method captures magnitude of price movement rather than just direction.

#### 📌 Important Note:
This notebook contains *interactive charts generated using Vectorbt.  
GitHub does not display interactive Plotly charts, so the graphs will not be visible here.  

✅ To view the charts, please download this notebook and run it on your local machine.  
Make sure you have Vectorbt and its dependencies installed to regenerate the visualizations.


## Part 1: Data & Feature Engineering

**Objective:**  
Load raw price data (MetaTrader 5 or CSV) and transform it into a feature-rich dataset.

**Tasks:**
- Fetch historical bars  
- Apply `ta.add_all_ta_features` or custom features  
- (Optionally) create specific labels (multi-bar, double-barrier, regime, etc.)  
- Clean/prepare the final feature matrix **X** and target **y**  

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Assuming the notebook is inside the notebooks folder, set the project root as one level up.
project_root = Path.cwd().parent
os.chdir(project_root)



import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt

# Our modules for data and backtesting
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import calculate_future_returns
from backtests.simple_backtest import simulate_trading, calculate_sharpe_ratio
from models.model_training import walk_forward_splits
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Deep Learning libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv1D, GlobalMaxPooling1D, LSTM
from tensorflow.keras.optimizers import Adam

# Set the project root (assuming the notebook is in a subfolder)
project_root = Path.cwd().parent
os.chdir(project_root)

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=1000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# Prepare features and labels
X = df.drop(columns=["future_returns"])
y = df["future_returns"]

###########################################################
# 2) WALK-FORWARD SPLITS
###########################################################

folds = walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds created: {len(folds)}")

###########################################################
# 3) HELPER FUNCTION TO CREATE SEQUENCES
###########################################################
def create_sequences(X, y, lookback):
    X_seq, y_seq = [], []
    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

# Define lookback window for sequence-based models
lookback = 10

###########################################################
# 4) DEFINE DL MODEL CONSTRUCTORS
###########################################################
# Model 1: MLP (Feed-Forward) that flattens the sequence
def create_mlp_model_seq(input_shape):
    model = Sequential([
         Flatten(input_shape=input_shape),
         Dense(64, activation='relu'),
         Dropout(0.2),
         Dense(32, activation='relu'),
         Dense(1)  # Regression output for future returns
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model

# Model 2: CNN for sequence data
def create_cnn_model_seq(input_shape):
    model = Sequential([
         Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
         GlobalMaxPooling1D(),
         Dense(64, activation='relu'),
         Dropout(0.2),
         Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model

# Model 3: LSTM for sequence data
def create_lstm_model_seq(input_shape):
    model = Sequential([
         LSTM(50, input_shape=input_shape),
         Dense(32, activation='relu'),
         Dropout(0.2),
         Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model

# Dictionary of DL model constructors
dl_models = {
    "MLP": create_mlp_model_seq,
    "CNN": create_cnn_model_seq,
    "LSTM": create_lstm_model_seq
}

###########################################################
# 5) DL MODEL TRAINING & BACKTESTING ACROSS MODELS
###########################################################
threshold = 0.0005  # Trade signal threshold
cost = 0.0002       # Transaction cost (0.02%)

fold_results = {}

for fold_i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n===== Fold {fold_i} =====")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_test_scaled = scaler.transform(X_test_fold)
    
    # Create sequences from scaled data
    # Note: This reduces the number of samples by 'lookback'
    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_fold.values, lookback)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_fold.values, lookback)
    
    # Adjust indices for backtesting to align with the test sequences
    test_indices = X_test_fold.index[lookback:]
    
    fold_results[fold_i] = {}
    
    for model_name, create_model in dl_models.items():
        print(f"Training model: {model_name}")
        input_shape = X_train_seq.shape[1:]  # (lookback, n_features)
        model = create_model(input_shape)
        model.fit(X_train_seq, y_train_seq, epochs=50, batch_size=32, verbose=0)
        
        preds = model.predict(X_test_seq).flatten()
        mse = mean_squared_error(y_test_seq, preds)
        
        # Convert predictions into trading signals
        signals = np.where(preds > threshold, 1, np.where(preds < -threshold, -1, 0))
        
        # Get corresponding rows in the original dataframe for backtesting
        df_test_fold = df.loc[test_indices].copy()
        
        # Run backtest
        daily_returns, total_return = simulate_trading(signals, df_test_fold, cost=cost)
        sr = calculate_sharpe_ratio(np.array(daily_returns))
        
        fold_results[fold_i][model_name] = {
             "MSE": mse,
             "TotalReturn": total_return,
             "Sharpe": sr
        }

###########################################################
# 6) PRINT RESULTS
###########################################################
for fold_i, models_dict in fold_results.items():
    print(f"\n=== Fold {fold_i} Results ===")
    for model_name, stats in models_dict.items():
        mse = stats["MSE"]
        ret = stats["TotalReturn"]
        sr = stats["Sharpe"]
        print(f"{model_name}: MSE={mse:.2e}, Return={ret:.2f}%, Sharpe={sr:.2f}")


Number of folds created: 3

===== Fold 1 =====
Training model: MLP
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
Training model: CNN
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Training model: LSTM
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step

===== Fold 2 =====
Training model: MLP
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Training model: CNN
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Training model: LSTM
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step

===== Fold 3 =====
Training model: MLP
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Training model: CNN
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Training model: LSTM
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step

=== Fold 1 Results ===
MLP: MSE=1.85e+00, Return=6.89%, Sharpe=0.03
CNN: MSE=4.32e-02, Return=-0.88%, Sharpe=0.00
LSTM: MSE=2.35e-03, Return=23.91%, Sharpe=0.09

=== Fold 2 Results ===
MLP: MSE=2.14e-03, Return=0.84%, Sharpe=0.01
CNN: MSE=4.72e-02, Return=-1.64%, Sharpe=-0.00
LSTM: MSE=4.13e-03, Return=17.07%, Sharpe=0.07

=== Fold 3 Results ===
MLP: MSE=2.35e-03, Return=5.86%, Sharpe

## Part 2: Model Training & Hyperparameter Tuning

**Objective:**  
Train an ML model (e.g., RandomForest, XGBoost) on the engineered features to predict the chosen labels.

**Tasks:**
- Perform time-based or walk-forward splits  
- Select top features if desired (e.g., using RandomForest feature importance)  
- Use `RandomizedSearchCV` or `GridSearchCV` to find optimal hyperparameters  
- Save the best model pipeline (e.g., `best_rf_pipeline.pkl`) 

In [ ]:
# Assuming the notebook is inside the notebooks folder, set the project root as one level up.
from pathlib import Path
project_root = Path.cwd().parent
os.chdir(project_root)


In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
# Assuming the notebook is inside the notebooks folder, set the project root as one level up.
from pathlib import Path
project_root = Path.cwd().parent
os.chdir(project_root)

import numpy as np
import pandas as pd
import MetaTrader5 as mt5
import joblib
from itertools import product

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# Deep Learning libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Your modules for data and feature engineering
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import calculate_future_returns

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# Create feature matrix X and target vector y
X_full = df.drop(columns=["future_returns"])
y_full = df["future_returns"]

###########################################################
# 2) PREPARE THE TUNING PORTION AND CREATE SEQUENCE DATA
###########################################################
# Define lookback window (number of timesteps in each sequence)
lookback = 10

# Split the first 80% of the data for hyperparameter tuning
split_idx = int(len(X_full) * 0.8)
X_tune = X_full.iloc[:split_idx].values  # Convert to numpy array
y_tune = y_full.iloc[:split_idx].values

# Scale the features before creating sequences
scaler = StandardScaler()
X_tune_scaled = scaler.fit_transform(X_tune)

# Helper function to create sequence data for LSTM
def create_sequences(X, y, lookback):
    X_seq, y_seq = [], []
    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

X_tune_seq, y_tune_seq = create_sequences(X_tune_scaled, y_tune, lookback)
print(f"Tuning portion size (after sequence creation): {len(X_tune_seq)} samples")

###########################################################
# 3) TIME-SERIES CROSS-VALIDATION SETUP
###########################################################
tscv = TimeSeriesSplit(n_splits=3)

###########################################################
# 4) DEFINE THE LSTM MODEL FUNCTION
###########################################################
def build_lstm_model(units, dropout_rate, learning_rate):
    model = Sequential([
        LSTM(units, input_shape=(lookback, X_tune_seq.shape[2])),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mean_squared_error')
    return model

###########################################################
# 5) DEFINE HYPERPARAMETER SPACE
###########################################################
param_grid = {
    "units": [30, 50, 70],
    "dropout_rate": [0.1, 0.2, 0.3],
    "learning_rate": [1e-3, 1e-4],
    "epochs": [50, 100],
    "batch_size": [16, 32]
}

# Get all possible hyperparameter combinations
param_combinations = list(product(*param_grid.values()))
print(f"Total hyperparameter combinations: {len(param_combinations)}")

###########################################################
# 6) MANUAL HYPERPARAMETER TUNING
###########################################################
best_mse = float("inf")
best_model = None
best_params = None

for params in param_combinations:
    units, dropout_rate, learning_rate, epochs, batch_size = params
    print(f"\nTraining model with: Units={units}, Dropout={dropout_rate}, LR={learning_rate}, Epochs={epochs}, Batch={batch_size}")

    # Create model
    model = build_lstm_model(units, dropout_rate, learning_rate)

    # Perform cross-validation
    mse_scores = []
    for train_idx, test_idx in tscv.split(X_tune_seq):
        X_train_fold, X_test_fold = X_tune_seq[train_idx], X_tune_seq[test_idx]
        y_train_fold, y_test_fold = y_tune_seq[train_idx], y_tune_seq[test_idx]

        # Train model
        model.fit(X_train_fold, y_train_fold, epochs=epochs, batch_size=batch_size, verbose=0)

        # Evaluate on test set
        y_pred = model.predict(X_test_fold)
        mse = mean_squared_error(y_test_fold, y_pred)
        mse_scores.append(mse)

    avg_mse = np.mean(mse_scores)
    print(f"Avg MSE: {avg_mse:.6f}")

    # Track the best model
    if avg_mse < best_mse:
        best_mse = avg_mse
        best_model = model
        best_params = params

print("\nBest Model Found:")
print(f"Units={best_params[0]}, Dropout={best_params[1]}, LR={best_params[2]}, Epochs={best_params[3]}, Batch={best_params[4]}")
print(f"Best Avg MSE: {best_mse:.6f}")

###########################################################
# 7) SAVE THE BEST MODEL
###########################################################
best_model.save("models/saved_models/best_lstm_model.h5")
print("Saved best LSTM model to 'models/saved_models/best_lstm_model.h5'")


Tuning portion size (after sequence creation): 1589 samples
Total hyperparameter combinations: 72

Training model with: Units=30, Dropout=0.1, LR=0.001, Epochs=50, Batch=16
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Avg MSE: 0.242755

Training model with: Units=30, Dropout=0.1, LR=0.001, Epochs=50, Batch=32
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Avg MSE: 0.012655

Training model with: Units=30, Dropout=0.1, LR=0.001, Epochs=100, Batch=16
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Avg MSE: 0.019566

Training model with: Units=30, Dropout=0.1, LR=0.001, Epochs=100, Batch=32
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Avg MSE: 0.013073

Training model with: Units=30, Dropout=0.1, LR=0.0

Avg MSE: 0.034152

Best Model Found:
Units=70, Dropout=0.2, LR=0.001, Epochs=100, Batch=16
Best Avg MSE: 0.003916
Saved best LSTM model to 'models/saved_models/best_lstm_model.h5'


## Part 3: Backtesting & Performance Evaluation

**Objective:**  
Evaluate how well the trained model performs on unseen data, simulating real trades.

**Tasks:**
- Use walk-forward or expanding splits to mimic “live” conditions  
- Convert model predictions to signals ([-1, 0, +1] or buy/sell/hold)  
- Run a simple backtest script or VectorBT for performance metrics  
- Calculate returns, Sharpe ratio, drawdowns, confusion matrix, etc.  
- Visualize results (equity curve, trades, etc.) to judge strategy viability  

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
# Assuming the notebook is inside the notebooks folder, set the project root as one level up.
from pathlib import Path
project_root = Path.cwd().parent
os.chdir(project_root)

import numpy as np
import pandas as pd
import MetaTrader5 as mt5
import vectorbt as vbt
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.optimizers import Adam  # Import Adam optimizer

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import calculate_future_returns

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# If df's index is date/time, ensure it's sorted chronologically
df = df.sort_index()

X = df.drop(columns=["future_returns"])
y = df["future_returns"]

###########################################################
# 2) DEFINING EXPANDING WALK-FORWARD SPLITS
###########################################################
def expanding_walk_forward_splits(X, y, lookback=10, n_splits=3):
    """
    Creates expanding walk-forward folds.
    Ensures X_train/X_test are correctly transformed into LSTM sequences.
    """
    n = len(X)
    fold_size = n // (n_splits + 1)
    splits = []
    
    for i in range(n_splits):
        train_end = (i+1) * fold_size
        if i == n_splits - 1:
            test_end = n
        else:
            test_end = (i+2) * fold_size
            if test_end > n:
                test_end = n

        X_train_fold = X.iloc[:train_end]
        y_train_fold = y.iloc[:train_end]
        
        X_test_fold = X.iloc[train_end:test_end]
        y_test_fold = y.iloc[train_end:test_end]
        
        splits.append((X_train_fold, y_train_fold, X_test_fold, y_test_fold))
    return splits

folds = expanding_walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds: {len(folds)}")

###########################################################
# 3) LOAD BEST LSTM MODEL & RUN EXPANDING WALK-FORWARD
###########################################################
# Load the best model
best_model = load_model("models/saved_models/best_lstm_model.h5")
print("Loaded best LSTM model from 'best_lstm_model.h5'")

# Recompile the model after loading
best_model.compile(optimizer=Adam(learning_rate=0.001), loss="mean_squared_error")

threshold = 0.0005  # min predicted return for a trade
fees = 0.0002       # 0.02% transaction cost per trade
lookback = 10       # Same lookback window used in training

fold_results = {}

# Standardize the entire dataset
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Helper function to create sequence data
def create_sequences(X, y, lookback):
    X_seq, y_seq = [], []
    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

for i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n=== Fold {i} ===")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # Scale training & test fold
    X_train_scaled = scaler.transform(X_train_fold)
    X_test_scaled = scaler.transform(X_test_fold)

    # Convert into sequences for LSTM
    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_fold.values, lookback)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_fold.values, lookback)

    # 3.1 Train the model (expanding window)
    best_model.fit(X_train_seq, y_train_seq, epochs=100, batch_size=16, verbose=0)

    # 3.2 Predict on the test portion
    preds = best_model.predict(X_test_seq)
    mse = mean_squared_error(y_test_seq, preds)

    # 3.3 Convert predictions => signals with threshold and flatten to 1D
    signals = np.where(preds > threshold, 1, np.where(preds < -threshold, -1, 0))
    signals = signals.flatten()

    # 3.4 Vectorbt backtest on the test portion
    df_test_fold = df.loc[X_test_fold.index].copy()  # Use same index as X_test_fold
    close_prices = df_test_fold["close"]

    # Pad signals if needed
    if len(signals) < len(close_prices):
        signals = np.append(signals, [0] * (len(close_prices) - len(signals)))

    signals_s = pd.Series(signals, index=close_prices.index)

    pf = vbt.Portfolio.from_signals(
        close_prices,
        entries=signals_s > 0,
        exits=signals_s < 0,
        init_cash=10000,
        freq='4H',
        fees=fees
    )

    total_return = pf.total_return()
    sharpe_ratio = pf.sharpe_ratio()

    print(f"Fold {i} MSE={mse:.2e}, Return={total_return:.2f}%, Sharpe={sharpe_ratio:.2f}")

    # Full stats
    print("\nVectorbt stats for Fold", i)
    print(pf.stats())

    # Optional: Plot
    fig = pf.plot()
    fig.show()
    
    # Store results
    fold_results[i] = {
        "MSE": mse,
        "TotalReturn": total_return,
        "Sharpe": sharpe_ratio
    }

###########################################################
# 4) PRINT FOLD RESULTS
###########################################################
for i, stats in fold_results.items():
    print(f"\nFold {i} => MSE={stats['MSE']:.2e}, Return={stats['TotalReturn']:.2f}%, Sharpe={stats['Sharpe']:.2f}")


Number of folds: 3


Loaded best LSTM model from 'best_lstm_model.h5'

=== Fold 1 ===
Train size: 499, Test size: 499
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step
Fold 1 MSE=1.25e-04, Return=0.11%, Sharpe=1.32

Vectorbt stats for Fold 1
Start                               2024-06-19 20:00:00
End                                 2024-09-10 20:00:00
Period                                 83 days 04:00:00
Start Value                                     10000.0
End Value                                  11115.841515
Total Return [%]                              11.158415
Benchmark Return [%]                         -11.161481
Max Gross Exposure [%]                            100.0
Total Fees Paid                              324.564157
Max Drawdown [%]                              11.015256
Max Drawdown Duration                  48 days 16:00:00
Total Trades                                         82
Total Closed Trades                                  81
Total Open Trades                                     1
Open 


=== Fold 2 ===
Train size: 998, Test size: 499
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Fold 2 MSE=1.08e-02, Return=0.16%, Sharpe=2.41

Vectorbt stats for Fold 2
Start                               2024-09-11 00:00:00
End                                 2024-12-03 00:00:00
Period                                 83 days 04:00:00
Start Value                                     10000.0
End Value                                  11595.798853
Total Return [%]                              15.957989
Benchmark Return [%]                          65.759487
Max Gross Exposure [%]                            100.0
Total Fees Paid                              332.802295
Max Drawdown [%]                                8.45277
Max Drawdown Duration                  47 days 20:00:00
Total Trades                                         81
Total Closed Trades                                  80
Total Open Trades                                     1
Open Trade PnL                              -163.435122


=== Fold 3 ===
Train size: 1497, Test size: 502
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Fold 3 MSE=2.86e-04, Return=-0.03%, Sharpe=-0.12

Vectorbt stats for Fold 3
Start                               2024-12-03 04:00:00
End                                 2025-02-24 16:00:00
Period                                 83 days 16:00:00
Start Value                                     10000.0
End Value                                   9698.365726
Total Return [%]                              -3.016343
Benchmark Return [%]                          -1.570446
Max Gross Exposure [%]                            100.0
Total Fees Paid                              228.518688
Max Drawdown [%]                              13.317761
Max Drawdown Duration                  81 days 08:00:00
Total Trades                                         59
Total Closed Trades                                  59
Total Open Trades                                     0
Open Trade PnL                                     


Fold 1 => MSE=1.25e-04, Return=0.11%, Sharpe=1.32

Fold 2 => MSE=1.08e-02, Return=0.16%, Sharpe=2.41

Fold 3 => MSE=2.86e-04, Return=-0.03%, Sharpe=-0.12
